In [ ]:
# ================================
# 🔧 Mount Google Drive
# ================================
from google.colab import drive
drive.mount('/content/drive')

import requests
import pandas as pd
from datetime import datetime
import time
import os


# ===========================================
# Helper: Convert date -> UNIX timestamp
# ===========================================
def to_unix(date_str):
    return int(datetime.strptime(date_str, "%Y-%m-%d").timestamp())


# ===========================================
# 🔥 Hacker News Scraping Function (WITH DATE)
# ===========================================
def scrape_hn(keywords, start_date, end_date):

    print(f"\n🔎 Scraping posts from {start_date} to {end_date}...")

    start_ts = to_unix(start_date)
    end_ts   = to_unix(end_date)

    base = "https://hn.algolia.com/api/v1/search"
    all_results = []

    for kw in keywords:
        print(f"   → Searching keyword: '{kw}'")

        params = {
            "query": kw,
            "tags": "story",
            "numericFilters": f"created_at_i>{start_ts},created_at_i<{end_ts}",
            "hitsPerPage": 1000
        }

        r = requests.get(base, params=params)
        data = r.json()

        if "hits" not in data:
            continue

        for item in data["hits"]:
            all_results.append({
                "keyword": kw,
                "title": item.get("title"),
                "hn_link": f"https://news.ycombinator.com/item?id={item.get('objectID')}",
                "created_at_i": item.get("created_at_i")
            })

        time.sleep(1)

    return pd.DataFrame(all_results)


# ===========================================
# 🔧 COMBINED KEYWORDS FROM BOTH EVENTS
# Timeline: 2024-07-01 → 2025-10-31
# ===========================================
keywords_all = [
    "amd", "advanced micro devices", "silo ai", "siloai",
    "amd acquisition", "ai acquisition", "silo ai acquisition",
    "openai", "amd openai", "ai infrastructure",
    "gpu partnership", "data center gpu", "amd deal"
]

df_new = scrape_hn(keywords_all, "2024-07-01", "2025-10-31")

print(f"\n📌 Total posts collected: {len(df_new)}")


# ===========================================
# 🔄 Convert UNIX timestamp → date
# ===========================================
df_new["date"] = pd.to_datetime(df_new["created_at_i"], unit="s").dt.date


# ===========================================
# 💾 Save directly (NO MERGE, NO SPLIT)
# ===========================================
save_path = "/content/drive/MyDrive/HN_AMD_Scraped.xlsx"
df_new.to_excel(save_path, index=False)

print("\n💾 Saved to:")
print(save_path)


Mounted at /content/drive

🔎 Scraping posts from 2024-07-01 to 2025-10-31...
   → Searching keyword: 'amd'
   → Searching keyword: 'advanced micro devices'
   → Searching keyword: 'silo ai'
   → Searching keyword: 'siloai'
   → Searching keyword: 'amd acquisition'
   → Searching keyword: 'ai acquisition'
   → Searching keyword: 'silo ai acquisition'
   → Searching keyword: 'openai'
   → Searching keyword: 'amd openai'
   → Searching keyword: 'ai infrastructure'
   → Searching keyword: 'gpu partnership'
   → Searching keyword: 'data center gpu'
   → Searching keyword: 'amd deal'

📌 Total posts collected: 4939

💾 Saved to:
/content/drive/MyDrive/HN_AMD_Scraped.xlsx


In [ ]:
# ============================================
# 📂 Load scraped file
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

source_path = "/content/drive/MyDrive/HN_AMD_Scraped.xlsx"
save_path   = "/content/drive/MyDrive/HN_AMD_Final.xlsx"

df = pd.read_excel(source_path)
print(f"Loaded {len(df)} rows.")


# ============================================
# 🔥 Extract FULL POST TEXT
# ============================================
def get_hn_post_text(hn_link):
    try:
        item_id = hn_link.split("=")[-1]

        api_url = f"https://hacker-news.firebaseio.com/v0/item/{item_id}.json"
        r = requests.get(api_url)
        data = r.json()

        if not data:
            return ""

        text_html = data.get("text")
        if text_html is None:
            return ""

        soup = BeautifulSoup(text_html, "html.parser")
        return soup.get_text(separator="\n")

    except:
        return ""


# ============================================
# 📝 Extract all post contents
# ============================================
post_texts = []

print("\n🔎 Extracting post contents...\n")
for i, row in df.iterrows():
    print(f" → {i+1}/{len(df)}  ID = {row['hn_link']}")
    content = get_hn_post_text(row["hn_link"])
    post_texts.append(content)
    time.sleep(0.5)

df["post"] = post_texts


# ============================================
# 💾 Save final file (KEEP DATE)
# ============================================
df_final = df[
    ["keyword", "title", "post", "created_at_i", "date"]
]

df_final.to_excel(save_path, index=False)

print("\n🎉 DONE — Final file saved:")
print(save_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 4939 rows.

🔎 Extracting post contents...

 → 1/4939  ID = https://news.ycombinator.com/item?id=40970560
 → 2/4939  ID = https://news.ycombinator.com/item?id=43071983
 → 3/4939  ID = https://news.ycombinator.com/item?id=42054449
 → 4/4939  ID = https://news.ycombinator.com/item?id=41630913
 → 5/4939  ID = https://news.ycombinator.com/item?id=40926648
 → 6/4939  ID = https://news.ycombinator.com/item?id=45490549
 → 7/4939  ID = https://news.ycombinator.com/item?id=44663074
 → 8/4939  ID = https://news.ycombinator.com/item?id=45155986
 → 9/4939  ID = https://news.ycombinator.com/item?id=41386667
 → 10/4939  ID = https://news.ycombinator.com/item?id=42283933
 → 11/4939  ID = https://news.ycombinator.com/item?id=45376605
 → 12/4939  ID = https://news.ycombinator.com/item?id=45546593
 → 13/4939  ID = https://news.ycombinator.com/item?id=41815268
 → 14/4939 

In [ ]:
# ============================================
# 🔀 Separate title-only vs title+post
# ============================================

# Title-only posts
df_title_only = df[df["post"] == ""]

# Title + post (full text)
df_title_post = df[df["post"] != ""]


# ============================================
# 💾 Save into separate files
# ============================================
title_only_path  = "/content/drive/MyDrive/HN_AMD_Title_Only.xlsx"
title_post_path  = "/content/drive/MyDrive/HN_AMD_Title_and_Post.xlsx"

df_title_only.to_excel(title_only_path, index=False)
df_title_post.to_excel(title_post_path, index=False)

print("✅ Saved files:")
print(title_only_path)
print(title_post_path)


✅ Saved files:
/content/drive/MyDrive/HN_AMD_Title_Only.xlsx
/content/drive/MyDrive/HN_AMD_Title_and_Post.xlsx
